# Mood × RPE recovery checks

Here I test whether the provisional model recovers a known mood × RPE interaction from simulated data. I first use clean ratings, then add copying and partial anchoring to scale starting values to examine how these affect the estimates and confidence intervals.

## 1. The provisional model

### Question and observation unit

Does the association between reward prediction errors and the next happiness rating depend on prior happiness within the same play?

Each observation is an interval between successive happiness ratings. A play contains 12 ratings, giving 11 intervals. The pre-task rating provides prior happiness for the first interval; each subsequent rating is the outcome of one interval and the prior rating for the next, except the final rating, which is only an outcome.

For each interval, I sum the certain rewards, chosen-gamble EVs and RPEs occurring between its two ratings. For this provisional model, I use equally weighted interval sums as a simple starting point for checking interaction recovery. This leaves recency weighting out of the initial specification.


### Within- and between-play happiness

I use $H$ for observed happiness ratings. In the simulations, $L$ denotes latent mood, from which reported ratings are generated.

To distinguish fluctuations within a play from differences between plays, I separate prior happiness into two components (Curran & Bauer, 2011). A rating of 60 could be unusually high for a play averaging 40, or unusually low for a play averaging 70.

For play $p$, I calculate the mean of the 11 ratings used as prior happiness: baseline through the second-to-last rating. The final rating is excluded because it serves only as an outcome.

$$
\bar H_p=\frac{1}{11}\sum_{k=1}^{11}H_{p,k},
\qquad
M^W_{p,k}=H_{p,k}-\bar H_p,
\qquad
M^B_p=\bar H_p-50.
$$

- $M^W$ (`mood_within`): prior happiness above or below that play’s average.
- $M^B$ (`mood_between`): the play’s average prior happiness relative to 50.

The two components add up to the midpoint-centred prior rating:

$$
H_{p,k}-50=M^W_{p,k}+M^B_p.
$$

My question concerns whether the RPE association changes when prior happiness is above or below the same play’s average. The within-play interaction is therefore the target. Including a separate between-play interaction allows plays with different average happiness to have different RPE slopes, without requiring that association to equal the within-play one.

I use 50 because it is the midpoint of the 0–100 scale. Changing this reference changes the interpretation of $\beta_{\mathrm{RPE}}$, but leaves the fitted values and interaction coefficients unchanged. Both happiness components remain in rating points; neither is divided by a standard deviation.

### Provisional model equation

$$
\begin{aligned}
H_{p,k+1} ={}& \alpha_p + \rho M^W_{p,k} \\
&+ \beta_{\mathrm{CR}}CR_{p,k}
+ \beta_{\mathrm{EV}}EV_{p,k}
+ \beta_{\mathrm{RPE}}RPE_{p,k} \\
&+ \theta_W M^W_{p,k}RPE_{p,k}
+ \theta_B M^B_pRPE_{p,k} \\
&+ \tau_k + \varepsilon_{p,k}.
\end{aligned}
$$

### Model terms

- $p$ identifies the play.
- $k$ identifies the interval, $k=1,\dots,11$.
- $H_{p,k+1}$ (`next_happiness`) is the happiness rating at the end of the interval.
- $\alpha_p$ is a separate intercept for each play, allowing plays to differ in overall happiness level after accounting for the other model terms.
- $\rho$ is the carry-over coefficient for prior happiness relative to the play mean, $M^W$ (`mood_within`). It describes the prior–next rating association when interval RPE is zero and may reflect more than mood persistence.
- $\beta_{\mathrm{CR}}$, $\beta_{\mathrm{EV}}$ and $\beta_{\mathrm{RPE}}$ are the reward weights for the interval sums (`CR`, `EV`, `RPE`). Each describes the difference in predicted next happiness per additional game point, holding the other predictors fixed. For RPE, this coefficient applies when prior happiness equals the play mean and that mean is 50.
- $\theta_W$ is the within-play interaction coefficient and my main target: the change in the RPE slope per happiness point above or below the play mean. Its predictor column is `within_x_RPE`.
- $\theta_B$ is the between-play interaction coefficient: the change in the RPE slope per point higher average prior happiness. Its predictor column is `between_x_RPE`.
- $\tau_k$ accounts for differences in average happiness across interval positions, such as ratings declining as the play progresses. It represents a position pattern shared across plays, rather than each participant’s own drift. Position 1 has a shift of zero; the other ten shifts are relative to it.
- $\varepsilon_{p,k}$ is the residual for the interval.

The between-play component ($M^B_p$) is constant within each play, so its additive contribution is absorbed by the play intercept ($\alpha_p$). The interaction term $M^B_pRPE_{p,k}$ varies as RPE changes between intervals, allowing its coefficient, $\theta_B$, to be estimated.

In this provisional model, only the RPE slope varies with prior happiness. CR and EV slopes are constant across mood levels.


### Interpreting the RPE interaction

The association between interval RPE and the next happiness rating depends on both happiness components:

$$
\text{RPE slope}
=
\beta_{\mathrm{RPE}}
+\theta_W M^W_{p,k}
+\theta_B M^B_p.
$$

For a given play, $M^B_p$ stays constant. The RPE slope changes with prior happiness within that play according to $\theta_W$.

For example, take a play whose prior ratings average 50, so $M^B_p=0$. With $\beta_{\mathrm{RPE}}=0.05$ and $\theta_W=0.001$:

| Prior happiness relative to the play mean | $M^W$ | RPE slope | Contribution of RPE = +20 |
|---|---:|---:|---:|
| 20 points below | −20 | 0.03 | +0.6 |
| At the mean | 0 | 0.05 | +1.0 |
| 20 points above | +20 | 0.07 | +1.4 |

These are the RPE contributions to the predicted rating, not the total predicted rating or its change from the prior rating. An RPE of −20 gives contributions of −0.6, −1.0 and −1.4 respectively.

In this example, positive $\theta_W$ makes the RPE slope steeper at higher prior happiness: positive RPEs contribute more positively and negative RPEs more negatively.

### Fitting the model

I estimate the coefficients with ordinary least squares (OLS), i.e. standard linear regression. The model allows each play its own intercept, $\alpha_p$. Subtracting each play’s mean from the outcome and every predictor column cancels this intercept out, allowing me to estimate the slopes without explicitly calculating the play intercepts. The slopes are the same as if I had included a separate intercept for every play (Sommet & Lipps, 2025).

The equation contains interaction products between the happiness components and RPE, so I calculate those products first and then subtract their play means. Subtracting the means first and multiplying afterwards would give different quantities. The interval-position indicators are transformed alongside the other predictors.

Errors from intervals within the same play can be correlated. I therefore use cluster-robust standard errors, allowing errors within plays to be related while treating different plays as independent. In these simulations, each play represents a different participant. The finite-sample adjustment excludes the play intercepts because uncertainty is grouped by play (Cameron & Miller, 2015). The 95% confidence intervals use a t distribution with the number of plays minus one degrees of freedom.

With only 11 intervals per play, combining play intercepts with the previous rating as a predictor can bias coefficient estimates (Hamaker & Grasman 2015; Nickell, 1981). I use simulations with known generating values to assess how well the model recovers the interaction despite this concern. I check whether estimates are close to the generating values (recovery), and whether the 95% confidence intervals contain them about 95% of the time (coverage).

## 2. Generate one clean play

I first generate one play with 30 reward trials and 12 happiness ratings, using the GBE rating schedule. In this clean example, reported happiness equals the simulated latent mood.

### Generating the happiness sequence

I generate each new rating from the previous rating, the intervening rewards and a new random disturbance. Each play has a known reference level, $\mu$, specified before generating the ratings. The carry-over coefficient $\rho$ determines how much of the previous rating’s departure from this reference carries into the next rating.

For example, with $\mu=50$ and $\rho=0.6$, a prior rating of 60 contributes $0.6(60-50)=6$ points above the reference to the next rating, before adding rewards and the random disturbance.

The generating reference $\mu$ differs from the observed play mean $\bar H_p$. I specify $\mu$ in advance and calculate $\bar H_p$ afterwards from the 11 generated ratings used as prior happiness. 

### Starting settings

I use these illustrative settings for the first clean play, with no interaction or interval-position effects.

| Setting | Code name | Value |
|---|---|---:|
| Generating reference, $\mu$ | `mu` | 50 |
| Pre-task rating | `H[0]` | 50 |
| Carry-over, $\rho$ | `rho` | 0.6 |
| CR weight | `beta_CR` | 0.02 |
| EV weight | `beta_EV` | 0.01 |
| RPE weight at the generating reference | `beta_RPE` | 0.05 |
| Within-play interaction, $\theta_W$ | `theta_W` | 0 |
| SD of the random disturbances | `innovation_sd` | 3 |
| Interval-position effects, $\tau_k$ | Not added in the generator | 0 |

The pre-task rating equals the generating reference. At each subsequent interval, I add an independent normally distributed disturbance with mean 0 and SD 3. Reported happiness equals latent mood in this clean example. I apply no rounding or clipping and inspect the generated rating range.

### Generate 30 reward trials

I generate a simple reward sequence with a 50% probability of choosing the gamble on each trial, independently of mood and the offered amounts. Each gamble has two equally likely outcomes.

I draw the offered amounts independently from uniform integer distributions:

- Certain reward: −50 to +50.
- Gamble win: +20 to +100.
- Gamble loss: −100 to −20.

I then calculate CR, chosen-gamble EV and RPE using the same definitions as in the earlier notebooks.

In [1]:
import numpy as np
import pandas as pd

# fixed seed so the simulated play is reproducible
rng = np.random.default_rng(20261024)

n_trials = 30

# integers() excludes the upper bound, so use 51 and 101
certain_value = rng.integers(-50, 51, size=n_trials)
win_value = rng.integers(20, 101, size=n_trials)
lose_value = -rng.integers(20, 101, size=n_trials)

# independently draw the choice and potential gamble outcome
chose_gamble = rng.random(n_trials) < 0.5
gamble_won = rng.random(n_trials) < 0.5

offered_ev = (win_value + lose_value) / 2
gamble_outcome = np.where(gamble_won, win_value, lose_value)
outcome = np.where(chose_gamble, gamble_outcome, certain_value)

synthetic_trials = pd.DataFrame({
    "trial": np.arange(1, n_trials + 1),
    "certain_value": certain_value,
    "win_value": win_value,
    "lose_value": lose_value,
    "chose_gamble": chose_gamble,
    "outcome": outcome,
    "CR": np.where(chose_gamble, 0, certain_value),
    "EV": np.where(chose_gamble, offered_ev, 0),
    "RPE": np.where(chose_gamble, outcome - offered_ev, 0),
})

synthetic_trials.head()

,trial,certain_value,win_value,lose_value,chose_gamble,outcome,CR,EV,RPE
0,1,31,65,-80,False,31,31,0.0,0.0
1,2,26,78,-82,False,26,26,0.0,0.0
2,3,-48,81,-75,False,-48,-48,0.0,0.0
3,4,-1,84,-22,True,84,0,31.0,53.0
4,5,5,96,-98,False,5,5,0.0,0.0


In [2]:
# certain choices contribute CR; gamble outcomes equal EV + RPE
np.testing.assert_allclose(
    synthetic_trials["outcome"],
    synthetic_trials["CR"]
    + synthetic_trials["EV"]
    + synthetic_trials["RPE"],
)

print("Reward terms reconstruct every outcome.")

Reward terms reconstruct every outcome.


### Group into rating intervals

I group the 30 trials into the 11 intervals defined by the GBE rating schedule, then sum CR, EV and RPE within each interval.

In [3]:
# number of completed trials at each rating; 0 is baseline
rating_boundaries = np.array([
    0, 3, 6, 9, 11, 14, 17, 20, 22, 25, 28, 30
])

interval_rows = []

# pair each boundary with the next one: (0, 3), (3, 6), ...
for index, (left, right) in enumerate(zip(
    rating_boundaries[:-1], rating_boundaries[1:]
)):
    # iloc[left:right] gives trials left+1 to right (1-indexed)
    events = synthetic_trials.iloc[left:right]

    interval_rows.append({
        "interval": index + 1,
        "start_boundary": left,
        "end_boundary": right,
        "n_trials": len(events),
        "CR": events["CR"].sum(),
        "EV": events["EV"].sum(),
        "RPE": events["RPE"].sum(),
    })

synthetic_intervals = pd.DataFrame(interval_rows)

assert len(synthetic_intervals) == 11
assert synthetic_intervals["n_trials"].sum() == 30

# check that interval reward totals match the trial-level totals
np.testing.assert_allclose(
    synthetic_intervals[["CR", "EV", "RPE"]].sum(),
    synthetic_trials[["CR", "EV", "RPE"]].sum(),
)

synthetic_intervals

,interval,start_boundary,end_boundary,n_trials,CR,EV,RPE
0,1,0,3,3,9,0.0,0.0
1,2,3,6,3,5,44.0,110.0
2,3,6,9,3,18,-42.5,117.5
3,4,9,11,2,23,0.0,0.0
4,5,11,14,3,-65,0.0,0.0
5,6,14,17,3,0,10.0,-43.0
6,7,17,20,3,19,0.0,0.0
7,8,20,22,2,0,23.5,-130.5
8,9,22,25,3,-1,4.5,37.5
9,10,25,28,3,-16,0.0,0.0


### Generate the 12 ratings

I generate each rating sequentially from the preceding rating, interval rewards and a new random disturbance, using the settings above.

In [4]:
# Settings
mu = 50.0
rho = 0.6 # carry-over
beta_CR = 0.02
beta_EV = 0.01
beta_RPE = 0.05
theta_W = 0.0
innovation_sd = 3.0

# reserve space for baseline and 11 subsequent ratings
H = np.empty(12)
H[0] = mu  # fixed baseline for this first example

mood_errors = rng.normal(0, innovation_sd, size=11)

for index, interval in enumerate(
    synthetic_intervals.itertuples(index=False)
):
    prior_deviation = H[index] - mu

    # next rating = mu + carry-over + reward effects + noise
    # RPE weight shifts with prior mood deviation (inactive while theta_W = 0)
    H[index + 1] = (
        mu
        + rho * prior_deviation
        + beta_CR * interval.CR
        + beta_EV * interval.EV
        + (beta_RPE + theta_W * prior_deviation) * interval.RPE
        + mood_errors[index]
    )

synthetic_play = synthetic_intervals.copy()
# H[:-1] are the ratings before each interval, H[1:] the ratings after
synthetic_play["prior_happiness"] = H[:-1]
synthetic_play["next_happiness"] = H[1:]
synthetic_play["mood_error"] = mood_errors

print("Baseline:", H[0])
print("Rating range:", H.min(), "to", H.max())

synthetic_play[
    ["interval", "CR", "EV", "RPE",
     "prior_happiness", "next_happiness", "mood_error"]
]

Baseline: 50.0
Rating range: 39.0318828948934 to 59.84355766892471


,interval,CR,EV,RPE,prior_happiness,next_happiness,mood_error
0,1,9,0.0,0.0,50.000000,54.255220,4.075220
1,2,5,44.0,110.0,54.255220,54.500925,-4.092207
2,3,18,-42.5,117.5,54.500925,59.843558,1.333002
3,4,23,0.0,0.0,59.843558,54.927406,-1.438728
4,5,-65,0.0,0.0,54.927406,53.625126,1.968683
5,6,0,10.0,-43.0,53.625126,48.387269,-1.737807
6,7,19,0.0,0.0,48.387269,46.649986,-2.762375
7,8,0,23.5,-130.5,46.649986,39.031883,-2.668109
8,9,-1,4.5,37.5,39.031883,43.276434,-2.042696
9,10,-16,0.0,0.0,43.276434,43.628848,-2.017012


In the first interval, prior happiness equals the generating reference, CR is 9 and EV and RPE are zero. The next rating is therefore $50 + 0.02(9) + 4.0752 = 54.2552$. All generated ratings fall within 0–100 without clipping.

### Construct the happiness components and interactions

I calculate the play mean from the 11 prior ratings, then construct the within- and between-play components and their RPE interactions. These are the predictors used by the fitted model.

In [5]:
play_mean = synthetic_play["prior_happiness"].mean()

synthetic_play["mood_within"] = (
    synthetic_play["prior_happiness"] - play_mean
)

# between-play component relative to the scale midpoint
synthetic_play["mood_between"] = play_mean - 50

# form products before any fixed-effects transformation
synthetic_play["within_x_RPE"] = (
    synthetic_play["mood_within"] * synthetic_play["RPE"]
)
synthetic_play["between_x_RPE"] = (
    synthetic_play["mood_between"] * synthetic_play["RPE"]
)

print("Known generating reference:", mu)
print("Observed mean of prior ratings:", play_mean)
print("Mean within-play deviation:",
      synthetic_play["mood_within"].mean())

Known generating reference: 50.0
Observed mean of prior ratings: 49.82969591012304
Mean within-play deviation: 3.2297397080004553e-15


### Put generation into functions

I put the reward and rating calculations into functions so I can generate repeated datasets. Using the original seed, I check that they reproduce the worked example.

In [6]:
def generate_reward_intervals(rng):
    """Simulate 30 trials and sum CR, EV and RPE within the 11 GBE rating intervals."""
    n_trials = 30

    certain = rng.integers(-50, 51, size=n_trials)
    win = rng.integers(20, 101, size=n_trials)
    lose = -rng.integers(20, 101, size=n_trials)

    chose_gamble = rng.random(n_trials) < 0.5
    gamble_won = rng.random(n_trials) < 0.5

    offered_ev = (win + lose) / 2
    gamble_outcome = np.where(gamble_won, win, lose)
    outcome = np.where(chose_gamble, gamble_outcome, certain)

    trials = pd.DataFrame({
        "CR": np.where(chose_gamble, 0, certain),
        "EV": np.where(chose_gamble, offered_ev, 0),
        "RPE": np.where(chose_gamble, outcome - offered_ev, 0),
    })

    boundaries = [0, 3, 6, 9, 11, 14, 17, 20, 22, 25, 28, 30]
    rows = []

    for index, (left, right) in enumerate(zip(
        boundaries[:-1], boundaries[1:]
    )):
        events = trials.iloc[left:right]

        rows.append({
            "interval": index + 1,
            "n_trials": len(events),
            "CR": events["CR"].sum(),
            "EV": events["EV"].sum(),
            "RPE": events["RPE"].sum(),
        })

    return pd.DataFrame(rows)

In [7]:
# Reproduce earlier example with original seed
check_intervals = generate_reward_intervals(
    np.random.default_rng(20261024)
)

np.testing.assert_allclose(
    check_intervals[["CR", "EV", "RPE"]],
    synthetic_intervals[["CR", "EV", "RPE"]],
)

print("Reward function reproduces the worked example 🥳")

Reward function reproduces the worked example 🥳


In [8]:
def simulate_play(
    rng,
    play_id,
    mu=50.0,
    rho=0.6,
    beta_CR=0.02,
    beta_EV=0.01,
    beta_RPE=0.05,
    theta_W=0.0,
    innovation_sd=3.0,
):
    """Simulate one play: rewards, then 12 ratings, then the within/between mood terms."""

    # rewards are drawn before the errors, as in the worked example (keeps the seed check valid)
    intervals = generate_reward_intervals(rng)

    H = np.empty(12)
    H[0] = mu
    errors = rng.normal(0, innovation_sd, size=11)

    for index, interval in enumerate(
        intervals.itertuples(index=False)
    ):
        prior_deviation = H[index] - mu

        H[index + 1] = (
            mu
            + rho * prior_deviation
            + beta_CR * interval.CR
            + beta_EV * interval.EV
            + (beta_RPE + theta_W * prior_deviation) * interval.RPE
            + errors[index]
        )

    result = intervals.copy()
    result["play_id"] = play_id
    result["known_mu"] = mu
    result["prior_happiness"] = H[:-1]
    result["next_happiness"] = H[1:]
    result["mood_error"] = errors

    # construct fitted predictors from the realised prior ratings
    play_mean = result["prior_happiness"].mean()
    result["mood_within"] = result["prior_happiness"] - play_mean
    result["mood_between"] = play_mean - 50
    result["within_x_RPE"] = result["mood_within"] * result["RPE"]
    result["between_x_RPE"] = result["mood_between"] * result["RPE"]

    return result

In [9]:
check_play = simulate_play(
    np.random.default_rng(20261024),
    play_id=1,
)

np.testing.assert_allclose(
    check_play[["prior_happiness", "next_happiness"]],
    synthetic_play[["prior_happiness", "next_happiness"]],
)

print("Complete function reproduces the worked example.")

Complete function reproduces the worked example.


### Generate a dataset

I combine independently generated plays into one dataset. Each play has a reference level drawn uniformly from 40–60. The function accepts a seed, a play count and an interaction strength.

In [10]:
def simulate_dataset(seed, n_plays=500, theta_W=0.0):
    """Draw one reference per play, then generate the plays in order.

    theta_W only changes the arithmetic, so the same seed gives identical references, rewards and mood errors at any theta_W.
    """
    rng = np.random.default_rng(seed)
    references = rng.uniform(40, 60, size=n_plays)

    return pd.concat([
        simulate_play(
            rng,
            play_id=play_id,
            mu=reference,
            theta_W=theta_W,
        )
        for play_id, reference in enumerate(references, start=1)
    ], ignore_index=True)

### First dataset: no interaction

I generate 500 independent plays with no mood × RPE interaction. The plays differ in their generating reference levels, while reward weights and carry-over are fixed across plays. I retain all generated plays and inspect the rating range.

In [11]:
# different seed from the single-play example
n_plays = 500
simulated_data = simulate_dataset(20261026, n_plays=n_plays, theta_W=0.0)  # no moderation

# prior + next together cover all 12 ratings in each play, for the range check
all_ratings = np.concatenate([
    simulated_data["prior_happiness"].to_numpy(),
    simulated_data["next_happiness"].to_numpy(),
])

print("Plays:", simulated_data["play_id"].nunique())
print("Intervals:", len(simulated_data))
print("Rating range:", all_ratings.min(), "to", all_ratings.max())
print("Ratings outside 0–100:", np.any(
    (all_ratings < 0) | (all_ratings > 100)
))

Plays: 500
Intervals: 5500
Rating range: 23.01241778943075 to 83.47320283602842
Ratings outside 0–100: False


In [12]:
# check that the same seed reproduces the complete dataset
pd.testing.assert_frame_equal(
    simulate_dataset(20261026, n_plays=500, theta_W=0.0),
    simulated_data,
)

## 3. Fit the clean null-interaction data

### Prepare the model matrix

I combine the six main predictors with the interval-position indicators, then subtract play means from every predictor and the outcome to account for play intercepts.

In [13]:
import statsmodels.api as sm

# mood_between has no main-effect column: it's constant within a play,
# so the play intercept absorbs it. Only its interaction with RPE varies.
predictor_names = [
    "mood_within",
    "CR", "EV", "RPE",
    "within_x_RPE",
    "between_x_RPE",
]

# represent interval position with indicators, using position 1 as reference
# omit one indicator to avoid redundancy with the play intercepts
position_terms = pd.get_dummies(
    simulated_data["interval"],
    prefix="position",
    drop_first=True,
    dtype=float,
)

X = pd.concat([
    simulated_data[predictor_names],
    position_terms,
], axis=1)

play_groups = simulated_data["play_id"]

# subtract each play's mean from the outcome and every predictor
# (play fixed effects, so no intercept column is needed)
X_within = X - X.groupby(play_groups).transform("mean")
y_within = (
    simulated_data["next_happiness"]
    - simulated_data.groupby("play_id")["next_happiness"].transform("mean")
)

assert np.isfinite(X_within.to_numpy()).all()
assert np.isfinite(y_within.to_numpy()).all()

print("Rows:", len(X_within))
print("Predictor columns:", X_within.shape[1])
# full column rank means the predictors permit separate coefficient estimates
print("Matrix rank:", np.linalg.matrix_rank(X_within.to_numpy()))

Rows: 5500
Predictor columns: 16
Matrix rank: 16


### First fit

I compare the six main coefficient estimates with their generating values. The position indicators remain in the fitted model but are omitted from this table.

In [14]:
null_fit = sm.OLS(y_within, X_within).fit()

# copied from simulate_play defaults: update here if they change
generating_values = pd.Series({
    "mood_within": 0.6, # rho, the carry-over
    "CR": 0.02,
    "EV": 0.01,
    "RPE": 0.05,
    "within_x_RPE": 0.0, # theta_W
    "between_x_RPE": 0.0,
})

recovery_table = pd.DataFrame({
    "generating_value": generating_values,
    "estimate": null_fit.params.loc[generating_values.index], # ignore position dummies
})

recovery_table["difference"] = (
    recovery_table["estimate"]
    - recovery_table["generating_value"]
)

recovery_table

,generating_value,estimate,difference
mood_within,0.60,0.528797,-0.071203
CR,0.02,0.018023,-0.001977
EV,0.01,0.007040,-0.002960
RPE,0.05,0.048921,-0.001079
within_x_RPE,0.00,-0.000059,-0.000059
between_x_RPE,0.00,-0.000070,-0.000070


Both interaction estimates are close to zero in this dataset. The estimated carry-over is 0.529, compared with the generating value of 0.6. I repeat the simulation below to check whether these differences persist across datasets.

### Reuse the fitting procedure

In [15]:
# uses predictor_names and generating_values from above; returns only the six main terms

def fit_simulated_dataset(data):
    """Fit the within-play model to one simulated dataset and return the six main estimates."""
    position_terms = pd.get_dummies(
        data["interval"],
        prefix="position",
        drop_first=True,
        dtype=float,
    )

    X = pd.concat([
        data[predictor_names],
        position_terms,
    ], axis=1)

    groups = data["play_id"]
    X_within = X - X.groupby(groups).transform("mean")
    y_within = (
        data["next_happiness"]
        - data.groupby("play_id")["next_happiness"].transform("mean")
    )

    # check that the columns permit separate coefficient estimates
    assert np.linalg.matrix_rank(X_within.to_numpy()) == X_within.shape[1]

    fit = sm.OLS(y_within, X_within).fit()
    return fit.params.loc[generating_values.index]

In [16]:
np.testing.assert_allclose(
    fit_simulated_dataset(simulated_data),
    recovery_table["estimate"],
)

### First 10 repetitions

I first run 10 datasets of 500 plays to check the repeated fitting procedure and its runtime. These runs are also included in the 200-repetition batch below.

In [17]:
from time import perf_counter

# spawn gives independent random streams from one base seed
# same base seed as the 200-run batch below: these are its first 10 runs
seeds = np.random.SeedSequence(2026102601).spawn(10)
simulation_results = []
started = perf_counter()

for repetition, seed in enumerate(seeds, start=1):
    # generate this repetition from its own seed
    data = simulate_dataset(seed, n_plays=500, theta_W=0.0)

    estimates = fit_simulated_dataset(data)

    simulation_results.append({
        "repetition": repetition,
        **estimates.to_dict(),
    })

    print(f"Finished repetition {repetition}/10")

null_recovery = pd.DataFrame(simulation_results).set_index("repetition")

print(f"Elapsed: {perf_counter() - started:.1f} seconds")

null_summary = pd.DataFrame({
    "generating_value": generating_values,
    "mean_estimate": null_recovery.mean(),
    "SD_across_runs": null_recovery.std(ddof=1),
})
null_summary["mean_difference"] = (
    null_summary["mean_estimate"] - null_summary["generating_value"]
)

# saved so the 200-run batch below can check it reproduces these
first_ten = null_recovery.copy()

null_summary

Finished repetition 1/10
Finished repetition 2/10
Finished repetition 3/10
Finished repetition 4/10
Finished repetition 5/10
Finished repetition 6/10
Finished repetition 7/10
Finished repetition 8/10
Finished repetition 9/10
Finished repetition 10/10
Elapsed: 10.4 seconds


,generating_value,mean_estimate,SD_across_runs,mean_difference
mood_within,0.60,0.520416,0.008692,-0.079584
CR,0.02,0.019751,0.001282,-0.000249
EV,0.01,0.008656,0.002489,-0.001344
RPE,0.05,0.049060,0.000385,-0.000940
within_x_RPE,0.00,0.000023,0.000111,0.000023
between_x_RPE,0.00,0.000044,0.000105,0.000044


Across these 10 datasets, mean estimated carry-over is 0.520, compared with 0.6 generated. Both interaction estimates average close to zero.

### Increase to 200 repetitions

I repeat the simulation 200 times with the same settings. Alongside the mean estimates and their variation across datasets, I calculate the Monte Carlo standard error (MCSE): how precisely these repetitions estimate the average fitted coefficient (Morris et al., 2019).

In [18]:
# same base seed as the 10-run check above: runs 1-10 should match it
seeds = np.random.SeedSequence(2026102601).spawn(200)
simulation_results = []
started = perf_counter()

for repetition, seed in enumerate(seeds, start=1):
    data = simulate_dataset(seed, n_plays=500, theta_W=0.0)

    estimates = fit_simulated_dataset(data)

    simulation_results.append({
        "repetition": repetition,
        **estimates.to_dict(),
    })

    if repetition % 20 == 0:
        print(f"Finished repetition {repetition}/{len(seeds)}")

null_recovery = pd.DataFrame(simulation_results).set_index("repetition")

# confirming the same base seed reproduces the first ten streams from before
# .loc includes both endpoints, so this selects repetitions 1–10
np.testing.assert_allclose(null_recovery.loc[1:10], first_ten) 

print(f"Elapsed: {perf_counter() - started:.1f} seconds")

null_summary = pd.DataFrame({
    "generating_value": generating_values,
    "mean_estimate": null_recovery.mean(),
    "SD_across_runs": null_recovery.std(ddof=1),
})

null_summary["mean_difference"] = (
    null_summary["mean_estimate"] - null_summary["generating_value"]
)

# MCSE: uncertainty in the mean estimate across runs = SD / sqrt(number of runs)
null_summary["MCSE_mean"] = (
    null_recovery.std(ddof=1) / np.sqrt(len(null_recovery))
)

null_summary

Finished repetition 20/200
Finished repetition 40/200
Finished repetition 60/200
Finished repetition 80/200
Finished repetition 100/200
Finished repetition 120/200
Finished repetition 140/200
Finished repetition 160/200
Finished repetition 180/200
Finished repetition 200/200
Elapsed: 184.8 seconds


,generating_value,mean_estimate,SD_across_runs,mean_difference,MCSE_mean
mood_within,0.60,0.526345,0.008696,-0.073655,0.000615
CR,0.02,0.019774,0.001122,-0.000226,0.000079
EV,0.01,0.009757,0.002264,-0.000243,0.000160
RPE,0.05,0.049217,0.000597,-0.000783,0.000042
within_x_RPE,0.00,0.000001,0.000125,0.000001,0.000009
between_x_RPE,0.00,-0.000002,0.000091,-0.000002,0.000006


Across 200 datasets, both interaction estimates average close to their generating value of zero. Mean carry-over is 0.526 rather than 0.6, and mean RPE weight is 0.0492 rather than 0.05. Recovery therefore differs between coefficients, with little average interaction bias under these clean null conditions.

## 4. Clean data with a nonzero interaction

I generate a dataset with $\theta_W=0.001$. The interaction depends on prior happiness relative to each play’s assigned reference, $\mu$. I fit the same ratings twice: first using this known reference, then using the observed play mean, as I would with real data.

### Why the two fits can differ

Rewards and random disturbances make the observed play mean differ from $\mu$. Expressing the interaction relative to that observed mean introduces an extra play-specific RPE slope. The between-play interaction cannot generally capture this, because it uses the play mean relative to 50, rather than relative to $\mu$.

For example, two plays could both average 52, giving them the same between-play score of $+2$. If their generating references are 50 and 54, their averages are respectively 2 points above and 2 points below those references. The extra RPE-slope terms are therefore $+2\theta_W$ and $-2\theta_W$, although their between-play predictors are identical.

If every play had $\mu=50$, the between-play interaction could represent this particular difference. Here, $\mu$ varies between plays, so I compare how the two reference choices affect the estimates.

### Interaction strength

I use 0.001 as an illustrative interaction strength. Ten happiness points above the generating reference increases the RPE slope by 0.01. For an interval RPE of +100, this adds 1 happiness point compared with being at the reference; an RPE of −100 gives the opposite contribution.

The size of this contribution depends on how much happiness varies within plays. I examine that variation later when interpreting the slider stress test.

In [19]:
# same seed preserves references, rewards and disturbances;
# changing theta_W changes the generated happiness sequence
nonzero_data = simulate_dataset(20261026, n_plays=500, theta_W=0.001)

# construct benchmark predictors using the known generating reference
nonzero_data["known_mood_within"] = (
    nonzero_data["prior_happiness"] - nonzero_data["known_mu"]
)
nonzero_data["known_mood_between"] = nonzero_data["known_mu"] - 50
nonzero_data["known_within_x_RPE"] = (
    nonzero_data["known_mood_within"] * nonzero_data["RPE"]
)
nonzero_data["known_between_x_RPE"] = (
    nonzero_data["known_mood_between"] * nonzero_data["RPE"]
)

# check the inputs match the null dataset
np.testing.assert_allclose(
    nonzero_data[["CR", "EV", "RPE", "mood_error", "known_mu"]],
    simulated_data[["CR", "EV", "RPE", "mood_error", "known_mu"]],
)

ratings = nonzero_data[
    ["prior_happiness", "next_happiness"]
].to_numpy()

print("Rating range:", ratings.min(), "to", ratings.max())

Rating range: 23.694961771684 to 84.40920589664164


In [20]:
# proposed model: mood relative to the observed play mean
observed_mean_estimates = fit_simulated_dataset(nonzero_data)

# benchmark: the fitting function always reads the same four column names,
# so swap those columns for their known-reference versions rather than change it
benchmark_data = nonzero_data.copy()  # copy so nonzero_data keeps the observed-mean columns

for field in [
    "mood_within",
    "mood_between",
    "within_x_RPE",
    "between_x_RPE",
]:
    benchmark_data[field] = nonzero_data[f"known_{field}"]  # e.g. known_mood_within

known_reference_estimates = fit_simulated_dataset(benchmark_data)

# generating coefficients for the known-reference specification
nonzero_generating_values = generating_values.copy()
nonzero_generating_values["within_x_RPE"] = 0.001

nonzero_comparison = pd.DataFrame({
    "generating_value_known_reference": nonzero_generating_values,
    "known_reference_fit": known_reference_estimates,
    "observed_mean_fit": observed_mean_estimates,
})

nonzero_comparison

,generating_value_known_reference,known_reference_fit,observed_mean_fit
mood_within,0.600,0.529059,0.529705
CR,0.020,0.018017,0.018023
EV,0.010,0.007048,0.006985
RPE,0.050,0.048908,0.048907
within_x_RPE,0.001,0.000970,0.000927
between_x_RPE,0.000,-0.000110,0.000143


In this dataset, the within-play interaction estimate is 0.000970 with the known reference and 0.000927 with the observed mean, compared with 0.001 generated. I repeat the paired comparison below to assess whether this difference persists across datasets.

### Repeat both reference choices 200 times

I generate 200 datasets with an interaction of 0.001, using the same random streams as the null simulations. For each dataset, I fit both reference specifications and retain the repetition number so I can compare their estimates on the same data.

In [21]:
# same base seed as the null batch
seeds = np.random.SeedSequence(2026102601).spawn(200)
nonzero_results = []
started = perf_counter()

for repetition, seed in enumerate(seeds, start=1):
    data = simulate_dataset(seed, n_plays=500, theta_W=0.001)

    # proposed observed-mean model
    observed_estimates = fit_simulated_dataset(data)

    # benchmark using the known generating reference
    benchmark = data.copy()
    benchmark["mood_within"] = (
        data["prior_happiness"] - data["known_mu"]
    )
    benchmark["mood_between"] = data["known_mu"] - 50
    benchmark["within_x_RPE"] = benchmark["mood_within"] * data["RPE"]
    benchmark["between_x_RPE"] = benchmark["mood_between"] * data["RPE"]

    known_estimates = fit_simulated_dataset(benchmark)

    # long format: one row per repetition per reference, so the two fits
    # stay paired by repetition
    for label, estimates in [
        ("known_reference", known_estimates),
        ("observed_mean", observed_estimates),
    ]:
        nonzero_results.append({
            "repetition": repetition,
            "reference": label,
            **estimates.to_dict(),
        })

    if repetition % 20 == 0:
        print(f"Finished {repetition}/{len(seeds)}")

nonzero_recovery = pd.DataFrame(nonzero_results)

print(f"Elapsed: {perf_counter() - started:.1f} seconds")

Finished 20/200
Finished 40/200
Finished 60/200
Finished 80/200
Finished 100/200
Finished 120/200
Finished 140/200
Finished 160/200
Finished 180/200
Finished 200/200
Elapsed: 187.6 seconds


In [22]:
interaction_summary = (
    nonzero_recovery.groupby("reference")["within_x_RPE"]
    .agg(mean_estimate="mean", SD_across_runs="std", runs="count")
)

interaction_summary["MCSE_mean"] = (
    interaction_summary["SD_across_runs"]
    / np.sqrt(interaction_summary["runs"]) # runs = number of repetitions
)
interaction_summary["difference_from_generating_value"] = (
    interaction_summary["mean_estimate"] - 0.001
)

interaction_summary

,mean_estimate,SD_across_runs,runs,MCSE_mean,difference_from_generating_value
reference,,,,,
known_reference,0.000985,0.000101,200,0.000007,-0.000015
observed_mean,0.000972,0.000124,200,0.000009,-0.000028


In [23]:
# put each dataset's two estimates side by side,
# then calculate the observed-minus-known difference within each repetition
paired_interactions = nonzero_recovery.pivot(
    index="repetition",
    columns="reference",
    values="within_x_RPE",
)

difference = (
    paired_interactions["observed_mean"]
    - paired_interactions["known_reference"]
)

print("Mean observed-minus-known difference:", difference.mean())
print("MCSE of that difference:",
      difference.std(ddof=1) / np.sqrt(len(difference)))

Mean observed-minus-known difference: -1.2553915557903172e-05
MCSE of that difference: 4.40312021033651e-06


Across 200 datasets, mean interaction estimates are 0.000985 with the known reference and 0.000972 with the observed play mean, compared with 0.001 generated. These are approximately 1.5% and 2.8% below the generating value.

Using the observed mean gives a lower estimate by 0.0000126 on average (MCSE 0.0000044), with greater variation between datasets.

## 5. Add confidence intervals

I extend the fitting function to return standard errors that allow errors within the same play to be related, and 95% confidence intervals. I first check that this leaves the coefficient estimates unchanged, then assess interval coverage across repeated datasets.

As in notebook 3, the adjustment excludes the play intercepts because uncertainty is grouped at the same or a broader level than the play effects (Cameron & Miller, 2015).

In [24]:
from scipy.stats import t

def fit_simulated_with_intervals(data):
    """Return the six main estimates, play-clustered SEs and 95% intervals."""
    positions = pd.get_dummies(
        data["interval"],
        prefix="position",
        drop_first=True,
        dtype=float,
    )

    X = pd.concat([data[predictor_names], positions], axis=1)
    groups = data["play_id"]

    X_within = X - X.groupby(groups).transform("mean")
    y_within = (
        data["next_happiness"]
        - data.groupby("play_id")["next_happiness"].transform("mean")
    )

    n = len(data)
    n_plays = groups.nunique()
    rank = np.linalg.matrix_rank(X_within.to_numpy())
    assert rank == X_within.shape[1]

    # fit with uncertainty clustered by play
    fit = sm.OLS(y_within, X_within).fit(
        cov_type="cluster",
        cov_kwds={"groups": groups, "use_correction": False}, # manual sample correction below
    )
    
    # finite-sample correction excluding play intercepts
    correction = (
        n_plays / (n_plays - 1)
        * (n - 1) / (n - rank)
    )

    # coefficient variances are on the covariance matrix diagonal
    se = pd.Series(
        np.sqrt(np.diag(fit.cov_params()) * correction),
        index=fit.params.index,
    )

    multiplier = t.ppf(0.975, df=n_plays - 1)

    return pd.DataFrame({
        "estimate": fit.params,
        "SE": se,
        "CI_low": fit.params - multiplier * se,
        "CI_high": fit.params + multiplier * se,
    }).loc[generating_values.index]

In [25]:
null_interval_fit = fit_simulated_with_intervals(simulated_data)

# check that the new function didn't change any coefficients
np.testing.assert_allclose(
    null_interval_fit["estimate"],
    fit_simulated_dataset(simulated_data),
)

null_interval_fit.loc[["within_x_RPE", "between_x_RPE"]]

,estimate,SE,CI_low,CI_high
within_x_RPE,-0.000059,0.000127,-0.000307,0.000190
between_x_RPE,-0.000070,0.000093,-0.000253,0.000114


Both interaction intervals include zero in this dataset. Repeated simulations below assess how often the intervals contain the generating value.

## 6. Slider stress test

### Fixed scenarios

These are illustrative settings. Slider starts are drawn independently and uniformly from integers 25–74, and copying is independent of mood and reward history.

- Clean: no measurement change.
- Copying: 25% of responses copy the slider start; the rest report latent mood.
- Copying + anchoring: the same copying decisions, with the other responses pulled 20% toward the start.

Each of the 12 latent ratings receives one reported value. Intermediate reported ratings are reused as the outcome of one interval and the prior rating of the next. The measurement layer changes reports while leaving the latent mood sequence unchanged. I retain all responses without rounding or clipping.

In [26]:
def add_slider_layer(latent_data, rng, copy_probability, anchor_strength):
    """Create reported ratings from latent ratings and rebuild model predictors."""
    plays = []

    for play_id, group in latent_data.groupby("play_id", sort=False):
        group = group.sort_values("interval").copy()
        assert np.array_equal(group["interval"], np.arange(1, 12))

        # reconstruct the 12-state sequence
        latent = np.r_[
            group["prior_happiness"].iloc[0],
            group["next_happiness"].to_numpy(),
        ]
        np.testing.assert_allclose(
            latent[:-1], group["prior_happiness"]
        )

        # draw 12 slider starts and copying decisions
        starts = rng.integers(25, 75, size=12)
        copies = rng.random(12) < copy_probability
        # always draw starts and copying numbers so paired conditions use the same draws
        # copying decisions also match when copy_probability is unchanged
        
        # copying responses report the start; other responses move toward it according to anchor_strength
        reported = np.where(
            copies,
            starts,
            (1 - anchor_strength) * latent + anchor_strength * starts,
        )

        # keep the underlying states for checking, not as fitted predictors
        group["prior_latent"] = latent[:-1]
        group["next_latent"] = latent[1:]
        group["prior_happiness"] = reported[:-1]
        group["next_happiness"] = reported[1:]
        group["prior_copied"] = copies[:-1]
        group["next_copied"] = copies[1:]

        # rebuild predictors from the observed ratings
        play_mean = reported[:-1].mean()
        group["mood_within"] = reported[:-1] - play_mean
        group["mood_between"] = play_mean - 50
        group["within_x_RPE"] = group["mood_within"] * group["RPE"]
        group["between_x_RPE"] = group["mood_between"] * group["RPE"]

        plays.append(group)

    return pd.concat(plays, ignore_index=True)

In [27]:
# with copying and anchoring off, ratings and model predictors should be unchanged
# predictors are rebuilt inside the function, so this also checks that rebuild

clean_check = add_slider_layer(
    simulated_data,
    np.random.default_rng(2026102801),
    copy_probability=0,
    anchor_strength=0,
)

np.testing.assert_allclose(
    clean_check[
        ["prior_happiness", "next_happiness"] + predictor_names
    ],
    simulated_data[
        ["prior_happiness", "next_happiness"] + predictor_names
    ],
)

print("Zero slider effects reproduce the clean data.")

Zero slider effects reproduce the clean data.


### Compare the six conditions

I run 100 repetitions of 500 plays for each combination of interaction strength (0 or 0.001) and measurement condition (clean, copying, or copying plus anchoring).

Within each repetition, the null and nonzero cases share references, rewards and random disturbances. The measurement conditions share slider-start draws; the two copying conditions also share copying decisions.

I assess recovery of the latent generating interaction and how often the confidence intervals contain it. I also count intervals excluding zero: false positives under the null, and detections under the nonzero condition.

In [28]:
# run three copy conditions x two theta conditions

# (copy_probability, anchor_strength)
conditions = {
    "clean": (0.0, 0.0),
    "copying": (0.25, 0.0),
    "copying_and_anchoring": (0.25, 0.20),
}

n_repetitions = 100
# generation seeds match the earlier batches, so these are the same latent datasets
generation_seeds = np.random.SeedSequence(2026102601).spawn(n_repetitions)
# separate seeds for the slider layer: starts and copying decisions
measurement_seeds = np.random.SeedSequence(2026102801).spawn(n_repetitions)

final_results = []
started = perf_counter()

for repetition, (generation_seed, measurement_seed) in enumerate(
    zip(generation_seeds, measurement_seeds), start=1
):
    for true_theta in (0.0, 0.001):
        # same generation seed for both theta values: identical rewards and mood errors
        latent_data = simulate_dataset(generation_seed, n_plays=500, theta_W=true_theta)

        for condition, (copy_probability, anchor_strength) in conditions.items():
            # reuse the measurement draws across conditions;
            # copying decisions match between the two conditions with 25% copying
            observed = add_slider_layer(
                latent_data,
                np.random.default_rng(measurement_seed),
                copy_probability=copy_probability,
                anchor_strength=anchor_strength,
            )

            estimates = fit_simulated_with_intervals(observed)
            interaction = estimates.loc["within_x_RPE"]

            # track the rating range, since nothing is clipped
            ratings = observed[
                ["prior_happiness", "next_happiness"]
            ].to_numpy()

            final_results.append({
                "repetition": repetition,
                "condition": condition,
                "true_theta": true_theta,
                "estimate": interaction["estimate"],
                "SE": interaction["SE"],
                "CI_low": interaction["CI_low"],
                "CI_high": interaction["CI_high"],
                "rho_estimate": estimates.loc["mood_within", "estimate"],
                "rating_min": ratings.min(),
                "rating_max": ratings.max(),
            })

    if repetition % 10 == 0:
        print(
            f"{repetition}/{n_repetitions} repetitions "
            f"({perf_counter() - started:.0f} seconds)"
        )

final_results = pd.DataFrame(final_results)

assert len(final_results) == 6 * n_repetitions
print(f"Finished in {perf_counter() - started:.1f} seconds")

10/100 repetitions (45 seconds)
20/100 repetitions (86 seconds)
30/100 repetitions (128 seconds)
40/100 repetitions (169 seconds)
50/100 repetitions (209 seconds)
60/100 repetitions (248 seconds)
70/100 repetitions (288 seconds)
80/100 repetitions (326 seconds)
90/100 repetitions (365 seconds)
100/100 repetitions (404 seconds)
Finished in 404.4 seconds


### Summarise recovery

For each condition, I report the mean interaction estimate, its difference from the generating value, interval coverage and the percentage of intervals excluding zero. Monte Carlo standard errors describe uncertainty from using a finite number of repetitions.

In [29]:
summary_rows = []

for (condition, true_theta), group in final_results.groupby(
    ["condition", "true_theta"]
):
    # coverage is against the latent theta, even for contaminated ratings
    covered = (
        (group["CI_low"] <= true_theta)
        & (group["CI_high"] >= true_theta)
    )
    # null: excluding zero = false positive; nonzero: = detection
    excludes_zero = (
        (group["CI_low"] > 0) | (group["CI_high"] < 0)
    )

    summary_rows.append({
        "condition": condition,
        "true_theta": true_theta,
        "runs": len(group),
        "mean_estimate": group["estimate"].mean(),
        "mean_difference": group["estimate"].mean() - true_theta,
        "MCSE_mean": group["estimate"].std(ddof=1) / np.sqrt(len(group)),
        "coverage_percent": 100 * covered.mean(),
        # MCSE of a proportion, in percentage points
        "coverage_MCSE_pp": (
            100 * np.sqrt(covered.mean() * (1 - covered.mean()) / len(group))
        ),
        "percent_excluding_zero": 100 * excludes_zero.mean(),
        "mean_rho": group["rho_estimate"].mean(),
        "rating_min": group["rating_min"].min(),
        "rating_max": group["rating_max"].max(),
    })

final_summary = pd.DataFrame(summary_rows)
final_summary

,condition,true_theta,runs,mean_estimate,mean_difference,MCSE_mean,coverage_percent,coverage_MCSE_pp,percent_excluding_zero,mean_rho,rating_min,rating_max
0,clean,0.000,100,-0.000008,-0.000008,0.000013,93.0,2.551470,7.0,0.525930,15.120124,81.603529
1,clean,0.001,100,0.000964,-0.000036,0.000013,91.0,2.861818,100.0,0.526630,18.292458,84.976946
2,copying,0.000,100,0.000003,0.000003,0.000022,97.0,1.705872,3.0,0.014590,17.163210,80.526251
3,copying,0.001,100,0.000144,-0.000856,0.000022,3.0,1.705872,10.0,0.015078,18.852934,84.976946
4,copying_and_anchoring,0.000,100,-0.000003,-0.000003,0.000023,97.0,1.705872,3.0,-0.020994,20.135535,78.247461
5,copying_and_anchoring,0.001,100,0.000089,-0.000911,0.000023,0.0,0.000000,10.0,-0.020655,21.968959,81.208167


### Mood variation relative to slider starts

The effect of copying depends partly on how much the slider starts vary relative to latent mood. I compare their variability using the two initial clean 500-play datasets.

I calculate the SD within each play, then report the median across plays. I show both the 11 prior ratings used to construct the predictors and the complete 12-rating sequence. For slider starts, I use the exact SD of the uniform distribution over integers 25–74.

I also express a one-SD difference in prior happiness as the corresponding change in RPE slope when $\theta_W=0.001$.

In [30]:
# Context for the stress test, using the two initial 500-play datasets.
variability_rows = []
slider_start_sd = np.arange(25, 75).std(ddof=0)  # exact uniform-distribution SD

for label, data in [("null", simulated_data), ("nonzero", nonzero_data)]:
    prior_sds = data.groupby("play_id")["prior_happiness"].std(ddof=1)
    all_rating_sds = []

    for _, play in data.groupby("play_id"):
        play = play.sort_values("interval")
        ratings = np.r_[
            play["prior_happiness"].iloc[0],
            play["next_happiness"].to_numpy(),
        ]
        all_rating_sds.append(ratings.std(ddof=1))

    variability_rows.append({
        "dataset": label,
        "plays": data["play_id"].nunique(),
        "median_SD_11_prior_ratings": prior_sds.median(),
        "median_SD_all_12_ratings": np.median(all_rating_sds),
        "uniform_start_SD": slider_start_sd,
        "slope_change_per_prior_SD_if_theta_0_001": 0.001 * prior_sds.median(),
    })

variability_summary = pd.DataFrame(variability_rows)
variability_summary.style.format({
    "median_SD_11_prior_ratings": "{:.2f}",
    "median_SD_all_12_ratings": "{:.2f}",
    "uniform_start_SD": "{:.2f}",
    "slope_change_per_prior_SD_if_theta_0_001": "{:.4f}",
}).hide(axis="index")

dataset,plays,median_SD_11_prior_ratings,median_SD_all_12_ratings,uniform_start_SD,slope_change_per_prior_SD_if_theta_0_001
null,500,4.51,4.56,14.43,0.0045
nonzero,500,4.51,4.58,14.43,0.0045


Slider starts vary much more than latent mood in this generator, helping explain why copying strongly disrupts recovery. These simulations use chosen mood-variation settings; they have not been calibrated to latent mood variability in GBE.

## 7. Conclusions, for now

With clean ratings, the model recovers the interaction fairly closely: estimates average near zero when no interaction is generated, and near the value I set when one is present. Carry-over is consistently underestimated, at about 0.53 compared with 0.6 generated. Clean-data confidence intervals cover the generating interaction in 93% of null runs and 91% of nonzero runs, with Monte Carlo standard errors of about 2.6 and 2.9 percentage points.

Adding copying makes the interaction much harder to recover. For the interaction strength I chose, 0.001, the mean estimate falls from 0.000964 with clean ratings to 0.000144 with copying and 0.000089 with copying plus anchoring. With the interaction present, coverage falls to 3% with copying, and none of the 100 intervals cover the generating value with copying plus anchoring. Neither measurement scenario produces much average interaction when none is generated.

The size of this deterioration depends on the settings. Slider starts vary much more than latent mood here, and copying is unrelated to mood or rewards. I can’t use these numbers to estimate how much the slider affects recovery in GBE.

The code recovers the interaction reasonably well in the clean examples, but recovering it from slider-affected ratings remains unresolved. The slider scenarios concern the GBE replication; the Wang et al. (2026) online samples had no default handle position.

## 8. References

Cameron, A. C., & Miller, D. L. (2015). A practitioner’s guide to cluster-robust inference. Journal of Human Resources, 50(2), 317–372. https://doi.org/10.3368/jhr.50.2.317

Curran, P. J., & Bauer, D. J. (2011). The disaggregation of within-person and between-person effects in longitudinal models of change. Annual Review of Psychology, 62, 583–619. https://doi.org/10.1146/annurev.psych.093008.100356

Hamaker, E. L., & Grasman, R. P. P. P. (2015). To center or not to center? Investigating inertia with a multilevel autoregressive model. Frontiers in Psychology, 5, Article 1492. https://doi.org/10.3389/fpsyg.2014.01492


Morris, T. P., White, I. R., & Crowther, M. J. (2019). Using simulation studies to evaluate statistical methods. Statistics in Medicine, 38(11), 2074–2102. https://doi.org/10.1002/sim.8086

Nickell, S. (1981). Biases in dynamic models with fixed effects. Econometrica, 49(6), 1417–1426. https://doi.org/10.2307/1911408

Sommet, N., & Lipps, O. (2025). A primer on fixed effects and fixed-effects panel modeling using R, stata, and SPSS. Advances in Methods and Practices in Psychological Science, 8(4), 25152459251392843. https://doi.org/10.1177/25152459251392843

Wang, Z., Wang, T., Xu, J., Aleman, A., Luo, Y., Blain, B., Liu, Y., & Xu, P. (2026). Dissociable Roles of Reward Prediction Error in the Contrasting Mood Dynamics of Depression and Anxiety. https://doi.org/10.7554/eLife.110631.2



